# ols_fe_ntv

> 对应代码：`EconometricsCode/code_in_notes/Chap.8/ols_fe_ntv.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.8`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.8")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

我们读入 NTV 自然实验的面板数据，并用 xtset 声明地区（tik_id）与年份，这样后续就可以使用 L.、L4. 等时间序列算子取滞后项。

In [ ]:
%%stata
clear
set more off
use ../datasets/NTV_Aggregate_Data_reshaped.dta
xtset tik_id year

我们用一个循环生成对数人口和对数工资的 1 至 5 阶多项式项。高阶多项式能灵活地控制地区社会经济规模的非线性影响，循环写法避免逐条书写重复命令。

In [ ]:
%%stata
forvalues i=1/5{
    gen log_pop_`i'=logpop^`i'
    gen log_wage_`i'=log_wage^`i'
}

我们先估计 1999 年横截面上的简单回归：以统一俄罗斯党的得票率为被解释变量、NTV 可收视强度（probit 变换）为解释变量，只吸收地区（region）固定效应。这是不控制其他因素的基准相关关系。

In [ ]:
%%stata
// 简单回归
reghdfe Votes_Edinstvo_ Watch_probit if year==1999, absorb(region)

我们把滞后一期的社会经济变量（人口与工资的多项式、护士数、人均医生数）存入宏并加入回归。这些滞后控制刻画了地区选举前的经济社会发展水平，用来缓解由地区差异带来的遗漏变量偏误。

In [ ]:
%%stata
// 加入社会经济控制
local E_controls "L.log_pop_* L.log_wage_* L.nurses L.doctors_pc"
reghdfe Votes_Edinstvo_ Watch_probit `E_controls' if year==1999, absorb(region)

最后我们加入 1995 年（滞后四期）各政党得票率和投票率，控制地区既往的政治倾向，使 NTV 收视强度的识别更接近“在可比的地区之间”的比较。对比三个设定中 Watch_probit 系数的变化，即可评估遗漏变量对估计的影响。

In [ ]:
%%stata
// 加入1995年选举的变量
local X_controls "L4.Votes_KPRF_ L4.Votes_Yabloko_ L4.Votes_NDR_ L4.Votes_LDPR_ L4.Turnout"
reghdfe Votes_Edinstvo_ Watch_probit `E_controls' `X_controls' if year==1999, absorb(region)